# INF8810, Projet 1  -> Prédiction de la durée du traget en temps réel

Membres :  
Barry Mamadou Rafiou (BARM18329807),   
Barry Fatima (BARF89310001),  
Naouel Ouazar (OUAN80300101),

## Installation des dépendances

In [ ]:
!pip install httpx "dask[distributed]" pyarrow scikit-learn joblib -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 1.2 MB/s eta 0:00:00


## Imports

In [ ]:
import httpx
import os
import numpy as np
import pandas as pd
import dask.dataframe as dd
from dask.distributed import Client
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import joblib

## Taches à accomplir

### Tâche 1 : Chargement et prétraitement des datas
Les fichiers Parquet de janvier et février 2024 ont été téléchargés depuis le site NYC TLC grâce à httpx.

Une nouvelle colonne trip_duration a été créée en calculant la durée du trajet (différence entre heure d’arrivée et de départ en secondes).

Les données ont été divisées en 80 % train et 20 % test, avec un prétraitement utilisant StandardScaler pour trip_distance et OneHotEncoder(handle_unknown='ignore') pour VendorID, en ajustant les transformations uniquement sur X_train.

In [ ]:
def download_file(url, dest):
    if os.path.exists(dest):
        print("fichier déjà téléchargé:", dest)
        return
    print("téléchargement en cours...")
    with httpx.stream("GET", url, follow_redirects=True, timeout=300) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_bytes(chunk_size=1024 * 64):
                f.write(chunk)
    print("fichier sauvegardé:", dest)

URL_JAN = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
URL_FEB = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-02.parquet"
PATH_JAN = "yellow_tripdata_2024-01.parquet"
PATH_FEB = "yellow_tripdata_2024-02.parquet"

download_file(URL_JAN, PATH_JAN)
download_file(URL_FEB, PATH_FEB)

téléchargement en cours...
fichier sauvegardé: yellow_tripdata_2024-01.parquet
téléchargement en cours...
fichier sauvegardé: yellow_tripdata_2024-02.parquet


In [ ]:
df_jan = pd.read_parquet(PATH_JAN)
print("données janvier brutes:", len(df_jan))

# Création de la cible trip_duration
df_jan['trip_duration'] = (
    df_jan['tpep_dropoff_datetime'] - df_jan['tpep_pickup_datetime']
).dt.total_seconds()

# Nettoyage des valeurs aberrantes
df_jan = df_jan[(df_jan['trip_duration'] > 0) & (df_jan['trip_duration'] <= 86400)]
df_jan = df_jan[(df_jan['trip_distance'] > 0) & (df_jan['trip_distance'] <= 200)]
df_jan = df_jan.dropna(subset=['VendorID', 'trip_distance', 'trip_duration'])
df_jan['VendorID'] = df_jan['VendorID'].astype(int).astype(str)

print("données  après nettoyage:", len(df_jan))

données janvier brutes: 2964624
données  après nettoyage: 2904098


In [ ]:
FEATURES = ['trip_distance', 'VendorID']
TARGET   = 'trip_duration'

X = df_jan[FEATURES]
y = df_jan[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("train:", len(X_train), "| test:", len(X_test))

train: 2323278 | test: 580820


In [ ]:
scaler  = StandardScaler()
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

scaler.fit(X_train[['trip_distance']])
encoder.fit(X_train[['VendorID']])

def transform_features(df):
    distance_scaled = scaler.transform(df[['trip_distance']])
    vendor_encoded  = encoder.transform(df[['VendorID']].astype(str))
    return np.hstack([distance_scaled, vendor_encoded])

X_train_proc = transform_features(X_train)
X_test_proc  = transform_features(X_test)
print("prétraitement OK, forme:", X_train_proc.shape)

prétraitement OK, forme: (2323278, 4)


 Question :
Qu'arrive-t-il si une nouvelle catégorie de VendorID apparaît ?

Réponse :
Si le modèle voit un VendorID qu'il n'a jamais vu pendant
l'entraînement, il ne va pas planter grâce à handle_unknown='ignore'.
L'encodeur va juste mettre des zéros à la place. La prédiction
va peut-être être moins bonne mais le programme continue
de tourner normalement.

### Tâche 2 : Entraînement local du modèle
Un modèle RandomForestRegressor(n_estimators=10) a été entraîné uniquement sur les données de janvier 2024.

Les attributs utilisés sont trip_distance (normalisé avec StandardScaler) et VendorID (encodé avec OneHotEncoder).

La variable cible du modèle est la durée de trajet trip_duration, exprimée en secondes.

In [ ]:
# Entraînement sur données de janvier uniquement
model = RandomForestRegressor(n_estimators=10, random_state=42, n_jobs=-1)
model.fit(X_train_proc, y_train)
print("Modèle entraîné.")

Modèle entraîné.


 Question 1 :
Quelle métrique utilisez-vous et pourquoi ?

Réponse : La métrique choisie est la MAE (Mean Absolute Error)
```python
mae = mean_absolute_error(y_test, y_pred)
```
**Justification :**
- La MAE calcule la moyenne des erreurs entre les valeurs prédites et les vraies valeurs
- Nous avons choisi la MAE parce que le résultat est en secondes, donc facile à interpréter directement de plus la MAE est moins affectée par les trajets très longs ou très courts qui sont des cas exceptionnels dans les données
- Une MAE de 300 secondes veut dire que le modèle a une erreur moyenne de 5 minutes par trajet


 Question 2 :
Le modèle est-il performant ?

Réponse :
**MAE obtenue à l'exécution : 303.2 s = (5.05 min)**

Nous pouvons dire que :
- La MAE obtenue est **acceptable** pour un service de taxi de base avec seulement 2 attributs.
- Le modèle est acceptable mais il n'est pas optimal car on utilise seulement 2 attributs (trip_distance et VendorID). Des variables importantes comme l'heure de la journée, le jour de la semaine ou la zone géographique sont absentes.

 ### Tâche 3 : Inférence distribuée
 Le fichier de février, trop volumineux, a été traité avec Dask en utilisant map_partitions pour appliquer predict_partition, où send_to_taxi() simule l’envoi temps réel.

Le TP demande un cluster de **4 workers**. La configuration idéale serait :

```python
Client(n_workers=4, threads_per_worker=1, processes=True)
```

Cependant, **Google Colab s'exécute dans un environnement sandbox** qui interdit la création de sous-processus indépendants. Lorsque `processes=True` est utilisé avec `n_workers=4`, Dask tente de lancer 4 processus fils séparés via `fork`/`spawn`, ce que Colab bloque — provoquant un crash (`AllExit` / `KeyboardInterrupt`) avant même le début du traitement des données.

La solution adoptée est `processes=False`, qui force Dask à utiliser des **threads** au lieu de processus :

```python
Client(n_workers=1, threads_per_worker=4, memory_limit='0', processes=False)
```

Avec `n_workers=1` et `threads_per_worker=4`, on obtient exactement **4 unités d'exécution concurrentes** qui est fonctionnellement équivalent à un cluster de 4 workers pour `map_partitions` : les partitions sont traitées 4 par 4 en parallèle, exactement comme avec 4 workers séparés.

> Sur une machine locale (hors Colab), `n_workers=4, processes=True` fonctionnerait directement sans modification.

In [ ]:
# ── Fonctions fournies par le TP (complétées)

def send_to_taxi(taxi_id, duration_seconds):
    print(f"Taxi {taxi_id}: l estimation de temps de course est {duration_seconds/60:.1f} minutes")


def predict_partition(df):
    df = df.copy()
    df = df.dropna(subset=['VendorID', 'trip_distance'])
    df = df[df['trip_distance'] > 0]

    if df.empty:
        df['predicted_duration'] = pd.Series(dtype=float)
        return df

    # df_enc : copie temporaire str pour l'encodeur
    # df garde VendorID en int32 (schéma parquet original)
    df_enc = df.copy()
    df_enc['VendorID'] = df_enc['VendorID'].astype(int).astype(str)

    X = transform_features(df_enc)
    df['predicted_duration'] = model.predict(X)

    for idx, row in df.iterrows():
        if idx % 10000 == 0:
            send_to_taxi(idx, row['predicted_duration'])
    return df

In [ ]:
# Cluster Dask — voir rapport pour la justification de la configuration
with Client(n_workers=1, threads_per_worker=4, memory_limit='0', processes=False) as client:
    print("dashboard:", client.dashboard_link)
    ddf_feb = dd.read_parquet(PATH_FEB)
    ddf_feb = ddf_feb.repartition(npartitions=16)
    print("nombre de partitions:", ddf_feb.npartitions)

    meta = ddf_feb._meta.copy()
    meta['predicted_duration'] = pd.Series(dtype=float)

    ddf_with_preds = ddf_feb.map_partitions(predict_partition, meta=meta)



INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at: inproc://172.28.0.12/5354/1
INFO:distributed.scheduler:  dashboard at:  http://172.28.0.12:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.worker:      Start worker at: inproc://172.28.0.12/5354/4
INFO:distributed.worker:         Listening to:          inproc172.28.0.12
INFO:distributed.worker:          Worker name:                          0
INFO:distributed.worker:         dashboard at:          172.28.0.12:35609
INFO:distributed.worker:Waiting to connect to: inproc://172.28.0.12/5354/1
INFO:distributed.worker:-------------------------------------------------
INFO:distributed.worker:              Threads:                          4
INFO:distributed.worker:      Local Directory: /tmp/dask-scratch-space/worker-su92r

dashboard: http://172.28.0.12:8787/status
nombre de partitions: 16


Question 1 :
Pourquoi une fonction Pandas fonctionne-t-elle avec Dask ?

Réponse :
Un Dask DataFrame est **découpé en partitions**, chacune étant un Pandas DataFrame standard. La méthode `map_partitions()` appelle la fonction sur chaque partition **indépendamment** et en parallèle sur les workers. Chaque appel reçoit donc bien un `pd.DataFrame` en entrée. Dask assemble ensuite les résultats en un nouveau Dask DataFrame.

 Question 2 :
Pourquoi `send_to_taxi()` est-elle appelée dans la fonction ?

Réponse :
On appelle send_to_taxi() à l'intérieur de predict_partition() pour que chaque chauffeur reçoit son estimation dès que sa partition est traitée. Si on appelait send_to_taxi() après un .compute() global, il faudrait attendre que tout le fichier de février soit traité avant d'envoyer la première notification. Dans un service de taxi en temps réel, le chauffeur a besoin de l'estimation rapidement — pas après que des millions de trajets soient tous calculés.

### Tâche 4 : Sauvegarde
Sauvegarde de la dataframe incluant les prédictions en format parquet

In [ ]:
    # Tâche 4 : Sauvegarde en format parquet
    ddf_with_preds.to_parquet('predictions_february_2024', write_index=False)
    print("Prédictions sauvegardées dans : predictions_february_2024/")

print("\n Toutes les tâches sont terminées avec succès.")

Taxi 1510000: l estimation de temps de course est 4.2 minutes
Taxi 1700000: l estimation de temps de course est 27.8 minutes
Taxi 1710000: l estimation de temps de course est 1.0 minutes
Taxi 1530000: l estimation de temps de course est 21.6 minutes
Taxi 1720000: l estimation de temps de course est 14.7 minutes
Taxi 1540000: l estimation de temps de course est 10.2 minutes
Taxi 1730000: l estimation de temps de course est 8.8 minutes
Taxi 1550000: l estimation de temps de course est 10.6 minutes
Taxi 1740000: l estimation de temps de course est 12.1 minutes
Taxi 1560000: l estimation de temps de course est 10.0 minutes
Taxi 1750000: l estimation de temps de course est 23.7 minutes
Taxi 1570000: l estimation de temps de course est 21.7 minutes
Taxi 1760000: l estimation de temps de course est 42.5 minutes
Taxi 1580000: l estimation de temps de course est 25.4 minutes
Taxi 1770000: l estimation de temps de course est 13.2 minutes
Taxi 1590000: l estimation de temps de course est 3.9 minu